# From sem1

In [1]:
import torch

def print_gpu_available_information():
    print("Number of GPU: ", torch.cuda.device_count())
    print("GPU Name: ", torch.cuda.get_device_name())

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Using device:', device)

print_gpu_available_information()

Number of GPU:  1
GPU Name:  NVIDIA GeForce RTX 3050 Ti Laptop GPU
Using device: cuda


## import

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow.feather as feather
import os
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import gc
import copy
import glob

In [6]:
# File path
CICIDS = 'NF-CICIDS2018-v3.csv'
UNSW = 'NF-UNSW-NB15-v3.csv'
insdn_files = glob.glob("../datasets/insdn/*.csv")

# Process in chunks
chunksize = 500000

In [8]:
for f in insdn_files:
    df = pd.read_csv(f)
    print(f"=== {f} ===")
    print("Columns:", df.columns.tolist())
    print("Shape:", df.shape)
    print("Label value counts:\n", df['Label'].value_counts(), "\n")

=== ../datasets/insdn\metasploitable-2.csv ===
Columns: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd

In [ ]:
from playground_utils import load_and_concatenate_datasets, inspect_missing_and_constant

df_raw = load_and_concatenate_datasets(insdn_files)
summary = inspect_missing_and_constant(df_raw)

# Print outputs
print("Columns with NaNs:")
print(summary['nan_count'][summary['nan_count'] > 0])

print("\nColumns with Infs:")
print(summary['inf_count'][summary['inf_count'] > 0])

print(f"\nConstant Columns ({len(summary['constant_columns'])}):")
print(summary['constant_columns'])

In [ ]:
# import pandas as pd

# # Load the preprocessed multiclass dataset
# df = pd.read_csv('processed/f52_imbalance_encode_multiclass.csv')

# # Show class distribution with label mapping
# label_map = {
#     0: 'BFA', 1: 'BOTNET', 2: 'DDoS', 3: 'DoS', 
#     4: 'Normal', 5: 'Probe', 6: 'U2R', 7: 'Web-Attack'
# }

# distribution = df['Label_Multi'].value_counts().sort_index()
# for label, count in distribution.items():
#     print(f"Label {label} ({label_map[label]}): {count} samples")

# Testing
model prediction code is below

In [5]:
# Import required libraries
import pandas as pd
import joblib
import xgboost as xgb
import numpy as np
import os

def get_feature_mappings():
    """Define feature mappings between CSV columns and model features"""
    # Features for each model (20 features version)
    features_20 = [
        'Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts',
        'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Flow Pkts/s',
        'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
        'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Max',
        'Bwd IAT Mean', 'Bwd IAT Min', 'Fwd Header Len',
        'Fwd Pkts/s', 'Pkt Len Max', 'Pkt Len Mean',
        'Init Bwd Win Byts'
    ]
    
    # CSV to model feature name mapping
    csv_to_model = {
        'Total Fwd Packet': 'Tot Fwd Pkts',
        'Bwd Packet Length Max': 'Bwd Pkt Len Max',
        'Bwd Packet Length Min': 'Bwd Pkt Len Min',
        'Flow Packets/s': 'Flow Pkts/s',
        'Fwd IAT Total': 'Fwd IAT Tot',
        'Fwd Header Length': 'Fwd Header Len',
        'Fwd Packets/s': 'Fwd Pkts/s',
        'Packet Length Max': 'Pkt Len Max',
        'Packet Length Mean': 'Pkt Len Mean',
        'Bwd Init Win Bytes': 'Init Bwd Win Byts'
    }
    
    return features_20, csv_to_model

def columnCheck(csv_path):
    """Check if CSV has required columns for prediction"""
    try:
        # Read CSV headers
        df = pd.read_csv(csv_path, nrows=0)
        csv_columns = set(df.columns)
        
        # Get feature mappings
        features_20, csv_to_model = get_feature_mappings()
        
        # For each required feature, check if either the original or mapped name exists
        missing_features = []
        for feature in features_20:
            # Get the CSV column name if it exists in the mapping
            csv_name = next((k for k, v in csv_to_model.items() if v == feature), feature)
            if feature not in csv_columns and csv_name not in csv_columns:
                missing_features.append(f"{feature} (or {csv_name})")
        
        if missing_features:
            print(f"Missing required columns: {missing_features}")
            return False
            
        print("All required columns are present!")
        return True
        
    except Exception as e:
        print(f"Error checking columns: {str(e)}")
        return False

def clean_features(df, columns):
    """Clean features by handling infinity and extreme values"""
    df_clean = df.copy()
    
    for col in columns:
        if col != 'Protocol':  # Skip categorical columns
            # Replace infinity with NaN
            df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
            
            # For each column, calculate reasonable bounds (e.g., 99th percentile)
            q99 = df_clean[col].quantile(0.99)
            q01 = df_clean[col].quantile(0.01)
            
            # Cap values at the bounds
            df_clean[col] = df_clean[col].clip(lower=q01, upper=q99)
            
            # Fill remaining NaN with median
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
    
    return df_clean

def predict(csv_path):
    """Make predictions using all three models and save results"""
    try:
        # Create output directory if it doesn't exist
        output_dir = "../output"
        os.makedirs(output_dir, exist_ok=True)
        
        # Get feature mappings
        features_20, csv_to_model = get_feature_mappings()
        
        # Load full data to preserve all columns
        df_full = pd.read_csv(csv_path)
        
        # Load scaler
        scaler = joblib.load("scalers/benign_robust_scaler.pkl")
        
        # Process features for prediction
        df_features = df_full.copy()
        
        # Ensure we have all features in the correct format
        for feature in features_20:
            csv_name = next((k for k, v in csv_to_model.items() if v == feature), feature)
            if csv_name in df_features.columns:
                df_features = df_features.rename(columns={csv_name: feature})
        
        # Clean features before scaling
        print("Cleaning features...")
        df_features = clean_features(df_features, features_20)
        
        # Scale features (excluding 'Protocol')
        scaled_cols = [col for col in features_20 if col != 'Protocol']
        print("Scaling features...")
        df_features[scaled_cols] = scaler.transform(df_features[scaled_cols])
        
        # Make predictions with each model
        models = {
            'predict20': 'best_xgb_20.json',
            'predict50': 'best_xgb_50.json',
            'predict80': 'best_xgb_80.json'
        }
        
        print("Making predictions...")
        for pred_col, model_file in models.items():
            print(f"Using model: {model_file}")
            model = xgb.XGBClassifier()
            model.load_model(f"models/{model_file}")
            
            # Get predictions
            X = df_features[features_20]
            
            # Additional safety check
            if X.isna().any().any():
                print(f"Warning: NaN values found in features. Filling with 0...")
                X = X.fillna(0)
            
            preds = model.predict(X)
            
            # Add predictions to full dataframe
            df_full[pred_col] = preds
        
        # Save results
        output_path = os.path.join(output_dir, f"{os.path.splitext(os.path.basename(csv_path))[0]}_predicted.csv")
        df_full.to_csv(output_path, index=False)
        
        # Print summary
        print(f"\nPrediction Summary for {os.path.basename(csv_path)}:")
        for pred_col in models.keys():
            attacks = sum(df_full[pred_col] == 1)
            total = len(df_full)
            print(f"{pred_col}: {attacks} attacks detected ({attacks/total*100:.2f}%)")
        
        print(f"\nResults saved to: {output_path}")
        return output_path
        
    except Exception as e:
        print(f"Error during prediction: {str(e)}")
        print("Stack trace:")
        import traceback
        traceback.print_exc()
        return None

In [6]:
csv_path = "../testDataSet/mirror.pcap_Flow.csv"
if columnCheck(csv_path):
    predict(csv_path)

All required columns are present!
Cleaning features...
Scaling features...
Making predictions...
Using model: best_xgb_20.json
Using model: best_xgb_50.json
Using model: best_xgb_80.json


All required columns are present!
Cleaning features...
Scaling features...
Making predictions...
Using model: best_xgb_20.json
Using model: best_xgb_50.json
Using model: best_xgb_80.json


c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\yuheng\AppData\Local\Temp\ipykernel_17380\3306041324.py:80: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[col] = df_clean[col].clip(lower=q01, upper=q99)


All required columns are present!
Cleaning features...
Scaling features...
Making predictions...
Using model: best_xgb_20.json
Using model: best_xgb_50.json
Using model: best_xgb_80.json


c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\yuheng\AppData\Local\Temp\ipykernel_17380\3306041324.py:80: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[col] = df_clean[col].clip(lower=q01, upper=q99)


Error during prediction: [Errno 13] Permission denied: '../output\\mirror.pcap_Flow_predicted.csv'
Stack trace:


Traceback (most recent call last):
  File "C:\Users\yuheng\AppData\Local\Temp\ipykernel_17380\3306041324.py", line 150, in predict
    df_full.to_csv(output_path, index=False)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\pandas\core\generic.py", line 3989, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        path_or_buf,
        ^^^^^^^^^^^^
    ...<14 lines>...
        storage_options=storage_options,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\pandas\io\formats\format.py", line 1014, in to_csv
    csv_formatter.save()
    ~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\pandas\io\fo

In [7]:
csv_path = "../testDataSet/mirror10min.pcap_Flow.csv"
if columnCheck(csv_path):
    predict(csv_path)

All required columns are present!


All required columns are present!


c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


All required columns are present!


c:\Users\yuheng\Documents\fyp\backend\fypenv\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Cleaning features...
Scaling features...
Making predictions...
Using model: best_xgb_20.json
Using model: best_xgb_50.json
Using model: best_xgb_80.json

Prediction Summary for mirror10min.pcap_Flow.csv:
predict20: 8582 attacks detected (33.55%)
predict50: 18849 attacks detected (73.68%)
predict80: 18284 attacks detected (71.47%)

Results saved to: ../output\mirror10min.pcap_Flow_predicted.csv

Prediction Summary for mirror10min.pcap_Flow.csv:
predict20: 8582 attacks detected (33.55%)
predict50: 18849 attacks detected (73.68%)
predict80: 18284 attacks detected (71.47%)

Results saved to: ../output\mirror10min.pcap_Flow_predicted.csv


In [5]:
# Check available columns in the CSV
import pandas as pd
df_check = pd.read_csv("../testDataSet/mirror.pcap_Flow.csv", nrows=1)
print("Available columns in CSV:")
for col in df_check.columns:
    print(col)

Available columns in CSV:
Flow ID
Src IP
Src Port
Dst IP
Dst Port
Protocol
Timestamp
Flow Duration
Total Fwd Packet
Total Bwd packets
Total Length of Fwd Packet
Total Length of Bwd Packet
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Bytes/s
Flow Packets/s
Flow IAT Mean
Flow IAT Std
Flow IAT Max
Flow IAT Min
Fwd IAT Total
Fwd IAT Mean
Fwd IAT Std
Fwd IAT Max
Fwd IAT Min
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Bwd IAT Min
Fwd PSH Flags
Bwd PSH Flags
Fwd URG Flags
Bwd URG Flags
Fwd Header Length
Bwd Header Length
Fwd Packets/s
Bwd Packets/s
Packet Length Min
Packet Length Max
Packet Length Mean
Packet Length Std
Packet Length Variance
FIN Flag Count
SYN Flag Count
RST Flag Count
PSH Flag Count
ACK Flag Count
URG Flag Count
CWR Flag Count
ECE Flag Count
Down/Up Ratio
Average Packet Size
Fwd Segment Size Avg
Bwd Segment Size Avg
Fwd Bytes/Bulk 

In [8]:
# Function to analyze prediction results
def analyze_predictions(csv_path, pred_column='predict20'):
    """
    Analyze prediction results with focus on ports and IP addresses
    Args:
        csv_path: Path to the predicted CSV file
        pred_column: Which prediction column to analyze (predict20/50/80)
    """
    # Load predictions
    df = pd.read_csv(csv_path)
    
    # Filter attacks
    attack_df = df[df[pred_column] == 1]
    
    print(f"\n=== Analysis for {pred_column} ===")
    print("\nDetailed Attack Records (showing first 10):")
    for _, row in attack_df.head(10).iterrows():
        print(f"Source: {row['Src IP']}:{row['Src Port']} -> Destination: {row['Dst IP']}:{row['Dst Port']}")
    
    print("\n=== Top 10 Source IPs and Ports in Attacks ===")
    src_counts = attack_df.groupby(['Src IP', 'Src Port']).size().sort_values(ascending=False).head(10)
    for (ip, port), count in src_counts.items():
        print(f"{ip}:{port} - {count} attacks")
    
    print("\n=== Top 10 Destination IPs and Ports in Attacks ===")
    dst_counts = attack_df.groupby(['Dst IP', 'Dst Port']).size().sort_values(ascending=False).head(10)
    for (ip, port), count in dst_counts.items():
        print(f"{ip}:{port} - {count} attacks")
    
    print("\n=== Most Common Attack Destination Ports ===")
    port_counts = attack_df['Dst Port'].value_counts().head(10)
    for port, count in port_counts.items():
        print(f"Port {port}: {count} attacks")
    
    print("\n=== Summary Statistics ===")
    total_flows = len(df)
    total_attacks = len(attack_df)
    print(f"Total flows analyzed: {total_flows}")
    print(f"Total attacks detected: {total_attacks}")
    print(f"Attack percentage: {(total_attacks/total_flows)*100:.2f}%")

# Analyze results for each model if prediction was successful
output_path = "../output/mirror.pcap_Flow_predicted.csv"
if os.path.exists(output_path):
    for pred_col in ['predict20', 'predict50', 'predict80']:
        analyze_predictions(output_path, pred_col)


=== Analysis for predict20 ===

Detailed Attack Records (showing first 10):
Source: 2.57.121.112:51442.0 -> Destination: 203.80.21.35:22.0
Source: 156.225.0.39:58914.0 -> Destination: 203.80.21.42:7834.0
Source: 156.225.0.10:58914.0 -> Destination: 203.80.21.41:23333.0
Source: 45.142.154.98:58914.0 -> Destination: 203.80.21.24:143.0
Source: 171.244.37.96:59738.0 -> Destination: 203.80.21.24:22.0
Source: 45.142.154.98:58914.0 -> Destination: 203.80.21.42:143.0
Source: 196.251.118.91:59701.0 -> Destination: 203.80.21.16:27017.0
Source: 91.126.49.0:58694.0 -> Destination: 203.80.21.24:22.0
Source: 156.225.0.17:58914.0 -> Destination: 203.80.21.35:8036.0
Source: 45.142.154.98:58914.0 -> Destination: 203.80.21.45:143.0

=== Top 10 Source IPs and Ports in Attacks ===
202.165.15.148:0.0 - 48 attacks
202.165.17.13:0.0 - 48 attacks
202.165.25.78:0.0 - 47 attacks
202.165.22.47:0.0 - 46 attacks
43.225.140.141:0.0 - 40 attacks
43.255.104.105:0.0 - 40 attacks
202.165.22.69:0.0 - 35 attacks
114.119

In [ ]:
# Test the updated prediction function
csv_path = "../testDataSet/mirror10min.pcap_Flow.csv"
if columnCheck(csv_path):
    predict(csv_path)